In [0]:
%sql
use catalog mobills;
with meses_calendario as (
    select distinct year_month
    from gold.calendar
),


trs as (
        select
            cal.year_month,
            SUM(trs.valor) AS valor,
            SUM(SUM(trs.valor)) OVER (ORDER BY cal.year_month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as acc_valor
        from gold.transacoes trs
        left join gold.calendar cal on trs.data = cal.calendar_date
        group by cal.year_month
),

orc as (
    select
        cal.year_month,
        sum(orc.saldo) as saldo,
        sum(sum(orc.saldo)) over(order by cal.year_month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as acc_saldo
    from gold.calendar cal 
    left join gold.orcamentos orc on cal.calendar_date = orc.data
    group by cal.year_month
),

final_cashflow as (
    select 
        cal.year_month,
        trs.acc_valor,
        coalesce(orc.acc_saldo, 0) as acc_saldo_orcamento,
        trs.acc_valor - coalesce(orc.acc_saldo, 0) as acc_fluxo
    from meses_calendario cal 
    left join trs on cal.year_month = trs.year_month
    left join orc on cal.year_month = orc.year_month 
)

select * from final_cashflow
order by year_month

year_month,acc_valor,acc_saldo_orcamento,acc_fluxo
2025-05-01,-5672.66,0.0,-5672.66
2025-06-01,-5987.5,0.0,-5987.5
2025-07-01,4035.469999999992,0.0,4035.469999999992
2025-08-01,-3734.840000000002,0.0,-3734.840000000002
2025-09-01,-707.3199999999938,0.0,-707.3199999999938
2025-10-01,-851.0499999999888,0.0,-851.0499999999888
2025-11-01,7023.050000000021,0.0,7023.050000000021
2025-12-01,427.26000000003296,0.0,427.26000000003296
2026-01-01,-2595.8999999999596,0.0,-2595.8999999999596
2026-02-01,-9111.529999999959,0.0,-9111.529999999959


In [0]:
(
    _sqldf
    .write
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable("mobills.gold.monthly_cashflow")
)